# 第40章：树模型

本 Notebook 复算正文第8节：一次切分、两轮提升、森林方差、置换重要性与 Shapley 小例，以及 5 倍信号敏感性面板上的模型比较。核心逻辑来自 `quant_course.tree_models` 与 `quant_course.ml_pipeline`。全部数据为**教学构造**（`data_class = CONSTRUCTED`），不是市场证据。

时间约定：signal_time $=D_k$ 收盘；trade_time $=D_{k+1}$ 收盘；标签区间 $D_{k+1}\to D_{k+6}$；label_available_at $=D_{k+6}$。训练行须 label_available_at $\le$ fit_time。

读者包离线路径：教学构造或明确标注的接受历史摘要。新运行输出位于唯一 `runs/` 子目录；不构成真实市场研究、P6验证或A10-B执行。


In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch40_tree_models_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def load_chapter(name):
    spec = importlib.util.spec_from_file_location(name, ROOT / "code" / f"{name}.py")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

from quant_course import ml_pipeline as mp, tree_models as tm
ch40 = load_chapter('ch40_tree_models')

## 1. 一次切分（第3节、第8.1节）

增益 $\Delta=\frac{n_Ln_R}{n}(\bar y_L-\bar y_R)^2$；sklearn 阈值取相邻取值中点。

In [ ]:
tab = pd.DataFrame(ch40.split_table(ch40.X8, ch40.Y8)).set_index("threshold")
display(tab.round(4))
assert np.allclose(tab["gain"], tab["gain_closed_form"]) and tab["gain"].idxmax() == 4.5 and tab["gain"].max() == 18.0
sk = ch40.sklearn_checks()
assert sk["stump_threshold"] == 4.5 and sk["stump_leaves"] == [-1.0, 2.0]
plt.bar(tab.index, tab["gain"], width=0.6, color="#247F82")
plt.xlabel("threshold s"); plt.ylabel("SSE reduction (%^2)"); plt.title("CONSTRUCTED 8-stock example"); plt.savefig(RUN_DIR / "ch40_tree_models_cell03.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 2. 两轮提升（学习率 0.5）与 L2 叶值（第8.3节）

In [ ]:
b = ch40.boosting_by_hand(ch40.X8, ch40.Y8, 0.5, 4)
display(pd.DataFrame([{k: r[k] for k in ("round", "train_mse")} | {"threshold": r.get("threshold")} for r in b]).round(5))
assert np.allclose(b[2]["F"], sk["hgb_round2_pred"], atol=1e-6)        # sklearn HGB reproduces F_2
assert sk["hgb_l2_leaf_check"] == [-0.25, 1.25]                        # leaf -6/(4+4) = -0.75 around F0 = 0.5
print("F2 =", b[2]["F"])

## 3. 森林方差、置换重要性与 Shapley（第8.2、8.4节）

In [ ]:
print({B: ch40.forest_variance(0.3, 1.0, B) for B in (1, 10, 100, float("inf"))})
assert np.isclose(ch40.forest_variance(0.3, 1.0, 10), 0.37) and np.isclose(ch40.forest_variance(0.1, 1.2, float("inf")), 0.12)
pt = ch40.permutation_toy()
print({k: round(v, 4) for k, v in pt.items() if k.startswith("imp")})
assert np.isclose(pt["imp_x1"], 1.56) and pt["imp_x2"] == 0.0
assert ch40.shapley_two(lambda a, b: 2 * a * b, (1.0, 1.0), (0.0, 0.0)) == (1.0, 1.0)
assert np.allclose(ch40.shapley_two(lambda a, b: 0.6 * a, (1.5, 1.5), (0.0, 0.0)), (0.9, 0.0))

## 4. 5 倍信号敏感性：交互面板上的七个模型（第8.6节，事后添加的敏感性，约 3–6 分钟）

登记的主结果（载荷 4e-4）中所有模型 IC 都在 ±0.01 内（第8.5节，CLI 复现）；这里重跑 5 倍面板以展示可检出时的差别。同一折、同一行、同一组合规则与第29章账本；零预测与训练均值空仓。

In [ ]:
close, labels, X = ch40.build_panel("interaction_strong")
folds = ch40.folds_for(labels)
out = ch40.run_models(close, labels, X, folds, importance=False)
rows = {m: {"R2_vs_mean_%": 100 * r["prediction"].get("oos_r2_vs_train_mean", np.nan), "IC": r["prediction"]["ic_mean"],
            "IC_t": r["prediction"]["ic_t"], "net10_SR": r["portfolio"]["0.001"]["sharpe"],
            "turnover": r["portfolio"]["0.001"]["turnover_ann"]} for m, r in out["metrics"].items()}
table = pd.DataFrame(rows).T
display(table.round(4))
assert round(table.loc["gbm", "IC"], 4) == 0.1190 and round(table.loc["ridge", "IC"], 4) == 0.0571
assert table.loc[["tree", "rf", "gbm"], "IC"].min() > table.loc[["ols", "ridge"], "IC"].max()
models = ["ols", "ridge", "tree", "rf", "gbm"]
plt.bar(models, table.loc[models, "IC"], color=["#7F7F7F", "#7F7F7F", "#247F82", "#247F82", "#247F82"])
plt.ylabel("mean daily Spearman IC (test)"); plt.title("CONSTRUCTED interaction panel, 5x signal"); plt.savefig(RUN_DIR / "ch40_tree_models_cell09.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 5. 读法

树模型在交互结构上 IC 约为线性模型的两倍；成本后夏普的绝对水平来自过强的构造信号，没有现实含义。主结果（登记强度）里树模型没有找到交互，这同样是本章要保留的结论。